# 10 — Conformal Prediction & Guaranteed Coverage

![Level](https://img.shields.io/badge/level-intermediate-yellow)
![Python](https://img.shields.io/badge/python-3.10%2B-blue)
![Twiga](https://img.shields.io/badge/twiga-forecast-orange)
![Time](https://img.shields.io/badge/time-~20%20min-lightgrey)

---

**What you'll build**

Coverage-guaranteed prediction intervals using Conformalised Quantile Regression (CQR) and Conformal Risk Control (CRC) applied as post-training wrappers on any trained Twiga model.

**Prerequisites**
- [08 — Quantile Regression](08-quantile-regression.ipynb) (QR models and interval metrics)
- [09 — Parametric Distributions](09-parametric-distributions.ipynb) (parametric probabilistic heads)
- Python: basic probability, coverage concepts

**Learning objectives**

By the end of this notebook you will be able to:

1. Explain the conformal prediction coverage guarantee and when it holds
2. Apply CQR and CRC as post-training wrappers to any Twiga model via `ConformalConfig`
3. Distinguish between CQR (tightens intervals) and CRC (adjusts a single threshold)
4. Evaluate coverage and sharpness trade-offs using PICP, NMPI, and Winkler score
5. Select the right conformal method based on your base model type

## 1. Motivation: Why Conformal Prediction?

The two probabilistic approaches covered in earlier notebooks each have a fundamental limitation:

- **Parametric (NB08)**: assumes data follows a known distribution family. Wrong family → miscalibrated coverage.
- **Quantile Regression (NB07)**: relies on asymptotic regime. Finite-sample coverage is not guaranteed.

> **Key concept — conformal prediction**
>
> Conformal prediction produces prediction intervals with a **finite-sample, distribution-free coverage guarantee**. Formally, for a significance level α ∈ (0, 1), it constructs intervals [l̂, û] such that:
>
> P(y_{n+1} ∈ [l̂_{n+1}, û_{n+1}]) ≥ 1 − α
>
> This is a **marginal coverage guarantee** — it holds over the randomness of the calibration and test draw, with no distributional assumption required. It works with *any* underlying model: tree ensembles, neural networks, or linear models.
>
> The key ingredient is a **calibration set** — a held-out set of labeled examples (separate from training) used to compute **nonconformity scores**, which measure how surprised the model is by each observation. The coverage guarantee follows directly from the exchangeability of calibration and test data.

> **Key concept — CQR vs. CRC**
>
> Twiga provides three conformal methods, differing in how they compute and adjust nonconformity scores:
>
> - **Split Conformal** (`method="residual"`): the simplest approach. Nonconformity score = |y − ŷ|. Interval = ŷ ± q_{1−α}(scores). Produces constant-width intervals — cannot adapt to local uncertainty.
> - **CQR — Conformal Quantile Regression** (`method="quantile"`): requires a QR base model. Adjusts the QR interval using calibration residuals. Inherits the adaptive width of the underlying QR model while adding the coverage guarantee. Best when you already have a QR model (NB07) and want sharper, locally adaptive intervals.
> - **CRC — Conformal Residual Calibration** (`method="residual-fitting"`): fits a secondary model to predict residual magnitude. Most expressive — interval width adapts to predicted local uncertainty. Best for heteroscedastic signals (e.g. solar-driven net load with strong day/night cycles).

> **Key concept — coverage vs. sharpness trade-off**
>
> All three methods satisfy the marginal coverage guarantee, but they differ in **sharpness** — how narrow the intervals are on average. Wider intervals are trivially easy to achieve (just make them very wide), so a good method is one that achieves the nominal coverage with the *smallest* possible interval width. This is measured by metrics like NMPI (Normalised Mean Prediction Interval width) and the Winkler score, which penalises both excess width and coverage violations jointly. The reliability diagram reveals whether a method is over- or under-covering systematically, while the sharpness comparison reveals which method produces the tightest intervals at the same nominal level.

In [ ]:
from great_tables import GT, md
import pandas as pd

from twiga.core.plot.gt import twiga_gt

df_methods = pd.DataFrame(
    {
        "Method": ["Split Conformal", "CQR", "CRC"],
        "Config method=": ['"residual"', '"quantile"', '"residual-fitting"'],
        "Key idea": [
            "Quantile of absolute residuals on the calibration set",
            "Adjust the QR interval using calibration residuals",
            "Fit a secondary model to predict residual magnitude adaptively",
        ],
        "Interval width": ["Constant", "Adaptive (from QR)", "Adaptive (from residual model)"],
    }
)

twiga_gt(
    GT(df_methods)
    .tab_header(
        title=md("**Conformal methods in Twiga**"),
        subtitle="All three methods satisfy the marginal coverage guarantee",
    )
    .cols_label(**{c: md(f"**{c}**") for c in df_methods.columns})
    .tab_source_note("Twiga Forecast"),
    n_rows=len(df_methods),
)

## 2. Setup

In [ ]:
import warnings

from great_tables import GT
from IPython.display import clear_output
from lets_plot import LetsPlot
import numpy as np
import pandas as pd
from sklearn.preprocessing import RobustScaler, StandardScaler

LetsPlot.setup_html()

from twiga.core.config import DataPipelineConfig, ForecasterConfig
from twiga.core.plot import (
    plot_forecast,
    plot_forecast_grid,
    plot_metrics_bar,
    plot_reliability_diagram,
)
from twiga.core.utils import configure, get_logger

warnings.filterwarnings("ignore")

configure()
log = get_logger("tutorials")

### Load data

The dataset covers Madeira, Portugal (32.37°N, 16.27°W) at 30-minute resolution.
We load only the columns needed: timestamp, net load (target), and two exogenous variables.

In [ ]:
data = pd.read_parquet("../data/MLVS-PT.parquet")
data = data[["timestamp", "NetLoad(kW)", "Ghi", "Temperature"]]
data["timestamp"] = pd.to_datetime(data["timestamp"])
data = data.drop_duplicates(subset="timestamp").reset_index(drop=True)

log.info("Shape: %s", data.shape)
GT(data.head())

### Train / val / test splits

We use the same fixed temporal split as all other tutorials. The **validation set doubles as the calibration set** — it is held out of training and used to compute nonconformity scores.

In [ ]:
from great_tables import GT, md
import pandas as pd

from twiga.core.plot.gt import twiga_gt

df_splits = pd.DataFrame(
    {
        "Split": ["train", "val / calibration", "test"],
        "Period": ["before 2021-01-01", "2021-01-01 – 2021-06-30", "2021-07-01 onwards"],
        "Role": [
            "Model training",
            "Early stopping + conformal calibration (nonconformity scores)",
            "Final evaluation of coverage and sharpness",
        ],
    }
)

twiga_gt(
    GT(df_splits)
    .tab_header(
        title=md("**Data splits**"),
        subtitle="Validation set is reused as the calibration set for all three conformal methods",
    )
    .cols_label(**{c: md(f"**{c}**") for c in df_splits.columns})
    .tab_source_note("Twiga Forecast"),
    n_rows=len(df_splits),
)

In [ ]:
train_df = data[data["timestamp"] < "2021-01-01"].reset_index(drop=True)
val_df = data[(data["timestamp"] >= "2021-01-01") & (data["timestamp"] < "2021-07-01")].reset_index(drop=True)
test_df = data[data["timestamp"] >= "2021-07-01"].reset_index(drop=True)

# The calibration set is the validation set
calibrate_df = val_df.copy()

log.info(
    f"train      : {train_df.shape[0]:,} rows  "
    f"({train_df['timestamp'].min().date()} → {train_df['timestamp'].max().date()})"
)
log.info(
    f"val        : {val_df.shape[0]:,} rows  ({val_df['timestamp'].min().date()} → {val_df['timestamp'].max().date()})"
)
log.info(
    f"test       : {test_df.shape[0]:,} rows  "
    f"({test_df['timestamp'].min().date()} → {test_df['timestamp'].max().date()})"
)
log.info(f"calibration: same as val — {calibrate_df.shape[0]:,} rows")

### Shared data and training configs

All three conformal methods share the same `DataPipelineConfig` and `ForecasterConfig`.
We define them once and reuse them throughout the notebook.

In [ ]:
data_config = DataPipelineConfig(
    target_feature="NetLoad(kW)",
    period="30min",
    latitude=32.371666,
    longitude=-16.274998,
    calendar_features=["hour", "day_night"],
    exogenous_features=["Ghi"],
    forecast_horizon=48,
    stride=48,
    lookback_window_size=96,
    input_scaler=StandardScaler(),
    target_scaler=RobustScaler(),
)

train_config = ForecasterConfig(
    split_freq="months",
    train_size=3,
    test_size=1,
)

data_config

## 3. Method 1 — Split Conformal (`method="residual"`)

Split conformal (also called inductive conformal prediction) is the simplest approach:

1. Train any point forecasting model on `train_df`.
2. Compute nonconformity scores $s_i = |y_i - \hat{y}_i|$ on the calibration set.
3. At test time, construct the interval as $\hat{y} \pm q_{1-\alpha}(s)$ where $q_{1-\alpha}$ is the $(1-\alpha)(1 + 1/n)$-quantile of the calibration scores.

**Pros**: works with any base model, fast calibration, finite-sample coverage guarantee.  
**Cons**: produces constant-width intervals — cannot adapt to regions of higher or lower uncertainty.

In [ ]:
from twiga import TwigaForecaster
from twiga.core.config import ConformalConfig
from twiga.models.ml import LIGHTGBMConfig

In [ ]:
# method="residual" corresponds to Split Conformal Prediction
conformal_config_split = ConformalConfig(method="residual", alpha=0.1)

forecaster_split = TwigaForecaster(
    data_params=data_config,
    model_params=[LIGHTGBMConfig()],
    train_params=train_config,
    conformal_params=conformal_config_split,
)
forecaster_split.fit(train_df=train_df, val_df=val_df)
clear_output()
log.info("Split Conformal — LightGBM training complete.")

In [ ]:
# Calibration step: computes nonconformity scores on the calibration set
forecaster_split.calibrate(calibrate_df=calibrate_df, ensemble_strategy="mean")
log.info("Calibration complete.")

In [ ]:
# predict_interval returns: dict[model_name -> (lower, forecast, upper)]
intervals_split, _ = forecaster_split.predict_interval(test_df=test_df)
log.info("Split conformal — available model keys: %s", list(intervals_split.keys()))

# Extract arrays for the first (and only) model
model_key_split = list(intervals_split.keys())[0]
lower_split, fc_split, upper_split = intervals_split[model_key_split]
log.info(f"lower shape: {lower_split.shape}, forecast shape: {fc_split.shape}, upper shape: {upper_split.shape}")

## 4. Method 2 — CQR: Conformal Quantile Regression (`method="quantile"`)

CQR (Angelopoulos & Bates, 2021) extends split conformal to models that already output quantile estimates:

1. Train a quantile regression model to output lower ($\hat{q}_{\alpha/2}$) and upper ($\hat{q}_{1-\alpha/2}$) quantiles.
2. Compute calibration nonconformity scores as $s_i = \max(\hat{q}_{\alpha/2} - y_i,\; y_i - \hat{q}_{1-\alpha/2})$.
3. Adjust the QR interval by adding the calibration quantile of these scores.

**Result**: CQR adapts the interval width to the local uncertainty estimated by the QR model, while still maintaining the marginal coverage guarantee. Intervals are typically **sharper** than split conformal.

In [ ]:
from twiga.models.ml import QRLIGHTGBMConfig

In [ ]:
# method="quantile" corresponds to Conformal Quantile Regression (CQR)
conformal_config_cqr = ConformalConfig(method="quantile", score_type="unscaled", alpha=0.1)

forecaster_cqr = TwigaForecaster(
    data_params=data_config,
    model_params=[QRLIGHTGBMConfig()],
    train_params=train_config,
    conformal_params=conformal_config_cqr,
)
forecaster_cqr.fit(train_df=train_df, val_df=val_df)
clear_output()
log.info("CQR — QR-LightGBM training complete.")

In [ ]:
forecaster_cqr.calibrate(calibrate_df=calibrate_df, ensemble_strategy="mean")
log.info("CQR calibration complete.")

In [ ]:
intervals_cqr, _ = forecaster_cqr.predict_interval(test_df=test_df)

model_key_cqr = list(intervals_cqr.keys())[0]
lower_cqr, fc_cqr, upper_cqr = intervals_cqr[model_key_cqr]
log.info(f"CQR arrays — lower: {lower_cqr.shape}, forecast: {fc_cqr.shape}, upper: {upper_cqr.shape}")

## 5. Method 3 — CRC: Conformal Residual Calibration (`method="residual-fitting"`)

CRC takes a different approach: instead of a fixed correction term, it **fits a secondary model** that predicts the magnitude of the residuals. This gives the interval an adaptive width — wider where the model is uncertain, narrower where it is confident.

CRC requires a base model that outputs both a **location** (point forecast) and a **scale** (predicted uncertainty). The `MLPGAMCRCConfig` model is purpose-built for this: it is a Group Additive Model with a conformal residual calibration head.

> **Note**: CRC is the most expressive of the three methods but also the most computationally demanding, since it involves training a neural network. We set `max_epochs=5` here for a quick demo.

In [ ]:
from twiga.models.nn import MLPGAMCRCConfig

In [ ]:
# method="residual-fitting" corresponds to Conformal Residual Calibration (CRC)
conformal_config_crc = ConformalConfig(method="residual-fitting", alpha=0.1)

crc_config = MLPGAMCRCConfig.from_data_config(data_config)
crc_config.max_epochs = 5
crc_config.rich_progress_bar = False

log.info("CRC model config ready.")

In [ ]:
forecaster_crc = TwigaForecaster(
    data_params=data_config,
    model_params=[crc_config],
    train_params=train_config,
    conformal_params=conformal_config_crc,
)
forecaster_crc.fit(train_df=train_df, val_df=val_df)
clear_output()
log.info("CRC — MLPGAMCRCConfig training complete.")

In [ ]:
forecaster_crc.calibrate(calibrate_df=calibrate_df, ensemble_strategy="mean")
log.info("CRC calibration complete.")

In [ ]:
intervals_crc, _ = forecaster_crc.predict_interval(test_df=test_df)

model_key_crc = list(intervals_crc.keys())[0]
lower_crc, fc_crc, upper_crc = intervals_crc[model_key_crc]
log.info(f"CRC arrays — lower: {lower_crc.shape}, forecast: {fc_crc.shape}, upper: {upper_crc.shape}")

## 6. Evaluating Coverage — PICP Comparison

We use `get_interval_metrics` to compute standard interval evaluation metrics for each method.

In [ ]:
from great_tables import GT, md
import pandas as pd

from twiga.core.plot.gt import twiga_gt

df_cov_metrics = pd.DataFrame(
    {
        "Metric": ["`picp`", "`ace`", "`nmpi`", "`winkle-score`", "`cwe`"],
        "Good value": ["≥ 1 − α", "≈ 0", "low", "low", "high"],
        "Interpretation": [
            "Prediction Interval Coverage Probability — must meet or exceed the nominal level",
            "Absolute Coverage Error — deviation from nominal; positive = under-coverage",
            "Normalised Mean Prediction Interval width — sharper is better at equal coverage",
            "Penalises both excess width and coverage violations jointly",
            "Combined Width-coverage Error (0–1, higher is better)",
        ],
    }
)

twiga_gt(
    GT(df_cov_metrics)
    .tab_header(
        title=md("**Coverage evaluation metrics**"),
        subtitle=md("From `twiga.core.metrics.get_interval_metrics` · nominal level = 90 % (α = 0.10)"),
    )
    .cols_label(**{c: md(f"**{c}**") for c in df_cov_metrics.columns})
    .tab_source_note("Twiga Forecast"),
    n_rows=len(df_cov_metrics),
)

In [ ]:
from twiga.core.metrics import get_interval_metrics

In [ ]:
def flatten_arrays(*arrays):
    """Flatten potentially multi-dimensional forecast arrays to 1-D."""
    return [a.reshape(-1) for a in arrays]


alpha = 0.1
results = []

for name, (lower, fc, upper) in [
    ("Split Conformal", (lower_split, fc_split, upper_split)),
    ("CQR", (lower_cqr, fc_cqr, upper_cqr)),
    ("CRC", (lower_crc, fc_crc, upper_crc)),
]:
    # We need actual values — use the test split ground truth
    # Ground truth shape must match forecast; align by min length
    fc_flat, lo_flat, hi_flat = flatten_arrays(fc, lower, upper)
    n = len(fc_flat)

    # Build ground truth from test_df target column (flattened to same length)
    true_vals = test_df["NetLoad(kW)"].values[:n]

    m = get_interval_metrics(
        pred=fc_flat[: len(true_vals)],
        true=true_vals,
        lower=lo_flat[: len(true_vals)],
        upper=hi_flat[: len(true_vals)],
        alpha=alpha,
    )
    m.insert(0, "Method", name)
    results.append(m)
    log.info("%s:", name)
    log.info("\n%s", m.to_string(index=False))

In [ ]:
summary = pd.concat(results, ignore_index=True)
log.info("\nSummary table (alpha=0.10, target coverage=90%):")
log.info("\n%s", summary.to_string(index=False))

## 7. Reliability Diagram Across Methods

A reliability diagram plots the **empirical coverage** (PICP) against the **nominal coverage** (1 − α) for a range of significance levels. A well-calibrated method should track the diagonal.

**How to read the reliability diagram:**
- Points on the diagonal → perfect calibration at that nominal level
- Points above the diagonal → over-coverage (intervals are unnecessarily wide; safe but wasteful)
- Points below the diagonal → under-coverage (the nominal guarantee is violated)
- All conformal methods should meet or exceed the diagonal by construction — deviations below indicate implementation issues or violation of the exchangeability assumption
- A method that is consistently above the diagonal is still valid but produces wider intervals than necessary; prefer methods that track the diagonal more closely

In [ ]:
alpha_values = np.linspace(0.05, 0.40, 15)

method_specs = [
    ("Split Conformal", lower_split, fc_split, upper_split),
    ("CQR", lower_cqr, fc_cqr, upper_cqr),
    ("CRC", lower_crc, fc_crc, upper_crc),
]

all_nominal = []
all_empirical = []
all_groups = []

for name, lower, fc, upper in method_specs:
    empirical_coverages = []
    for a in alpha_values:
        fc_f, lo_f, hi_f = flatten_arrays(fc, lower, upper)
        true_vals = test_df["NetLoad(kW)"].values[: len(fc_f)]
        m = get_interval_metrics(
            pred=fc_f[: len(true_vals)],
            true=true_vals,
            lower=lo_f[: len(true_vals)],
            upper=hi_f[: len(true_vals)],
            alpha=a,
        )
        empirical_coverages.append(float(m["picp"].iloc[0]))
    nominal_coverages = (1 - alpha_values).tolist()
    all_nominal.extend(nominal_coverages)
    all_empirical.extend(empirical_coverages)
    all_groups.extend([name] * len(nominal_coverages))

p = plot_reliability_diagram(
    nominal=all_nominal,
    empirical=all_empirical,
    group_col="Method",
    groups=["Split Conformal", "CQR", "CRC"],
    title="Reliability diagram — conformal methods",
)
p

## 8. Visualising Interval Width

We plot the prediction intervals from all three methods over the **first 48 steps (1 day)** of the test set.

Key observations to look for:
- **Split Conformal** produces constant-width bands.
- **CQR** inherits the adaptive shape of the underlying quantile model — typically sharper.
- **CRC** can vary width according to predicted local uncertainty from the secondary residual model.

In [ ]:
n_steps = 48  # 1 day at 30-min resolution

# Flatten and take first n_steps from each method
lo_s, fc_s, hi_s = [a.reshape(-1)[:n_steps] for a in (lower_split, fc_split, upper_split)]
lo_q, fc_q, hi_q = [a.reshape(-1)[:n_steps] for a in (lower_cqr, fc_cqr, upper_cqr)]
lo_c, fc_c, hi_c = [a.reshape(-1)[:n_steps] for a in (lower_crc, fc_crc, upper_crc)]
actual = test_df["NetLoad(kW)"].values[:n_steps]
steps = list(range(n_steps))

# Build combined DataFrame for plot_forecast_grid
split_df = pd.DataFrame({"Actual": actual, "forecast": fc_s, "lower": lo_s, "upper": hi_s, "Model": "Split Conformal"})
cqr_df = pd.DataFrame({"Actual": actual, "forecast": fc_q, "lower": lo_q, "upper": hi_q, "Model": "CQR"})
crc_df = pd.DataFrame({"Actual": actual, "forecast": fc_c, "lower": lo_c, "upper": hi_c, "Model": "CRC"})

all_intervals = pd.concat([split_df, cqr_df, crc_df], ignore_index=True)

p = plot_forecast_grid(
    all_intervals,
    actual_col="Actual",
    forecast_col="forecast",
    model_col="Model",
    n_samples_per_model=n_steps,
    title="Prediction intervals — first 48 steps of test set (1 day)",
    y_label="Net Load (kW)",
)
p

### Mean interval width comparison

In [ ]:
for name, lo, hi in [
    ("Split Conformal", lower_split.reshape(-1), upper_split.reshape(-1)),
    ("CQR", lower_cqr.reshape(-1), upper_cqr.reshape(-1)),
    ("CRC", lower_crc.reshape(-1), upper_crc.reshape(-1)),
]:
    widths = hi - lo
    log.info(f"{name:<20} mean width = {widths.mean():.2f}  std = {widths.std():.2f}")

## 9. When to Use Which Method

**General guidance:**
- Start with **Split Conformal** — it requires zero additional modelling effort and provides the coverage guarantee immediately with any point forecasting model.
- If you already have a quantile regression model (NB07), upgrade to **CQR** for free: it only adds a calibration step on top of your existing QR model and gives locally adaptive intervals.
- Use **CRC** when you have reason to believe uncertainty varies significantly across time (e.g. solar-driven net load, day/night cycles) and are willing to pay the cost of training a neural network backbone.

All three methods satisfy the marginal coverage guarantee under exchangeability. None requires specifying a parametric distribution family.

In [ ]:
from great_tables import GT, md
import pandas as pd

from twiga.core.plot.gt import twiga_gt

df_selector = pd.DataFrame(
    {
        "Method": ["Split Conformal", "CQR", "CRC"],
        "Config method=": ['"residual"', '"quantile"', '"residual-fitting"'],
        "Requires": [
            "Any point forecasting model",
            "A QR model (e.g. QRLIGHTGBMConfig)",
            "A location-scale model (e.g. MLPGAMCRCConfig)",
        ],
        "Best when": [
            "Fast, general-purpose guarantee; ideal starting point",
            "Want tighter, locally adaptive intervals on top of QR",
            "Heteroscedastic data; want residual model to drive width",
        ],
    }
)

twiga_gt(
    GT(df_selector)
    .tab_header(
        title=md("**Conformal method selector**"),
        subtitle="All three meet the marginal coverage guarantee under exchangeability",
    )
    .cols_label(**{c: md(f"**{c}**") for c in df_selector.columns})
    .tab_source_note("Twiga Forecast"),
    n_rows=len(df_selector),
)

---
## Interval Comparison

`plot_forecast_intervals` overlays Split Conformal, CQR, and CRC bands side by
side — immediately revealing differences in sharpness between methods.

In [ ]:
from twiga.core.plot import plot_forecast_intervals

n_plot = 48  # one day

# Flatten and slice all three methods to the same length
lo_s_f, fc_s_f, hi_s_f = [a.reshape(-1)[:n_plot] for a in (lower_split, fc_split, upper_split)]
lo_q_f, fc_q_f, hi_q_f = [a.reshape(-1)[:n_plot] for a in (lower_cqr, fc_cqr, upper_cqr)]
lo_c_f, fc_c_f, hi_c_f = [a.reshape(-1)[:n_plot] for a in (lower_crc, fc_crc, upper_crc)]

actual_f = test_df[data_config.target_feature].values[:n_plot]

cmp_df = pd.DataFrame(
    {
        "Actual": actual_f,
        "lo_s": lo_s_f,
        "hi_s": hi_s_f,
        "fc_s": fc_s_f,
        "lo_q": lo_q_f,
        "hi_q": hi_q_f,
        "fc_q": fc_q_f,
        "lo_c": lo_c_f,
        "hi_c": hi_c_f,
        "fc_c": fc_c_f,
    }
)

interval_specs = [
    {"model": "Split Conformal", "lower": "lo_s", "upper": "hi_s", "center": "fc_s"},
    {"model": "CQR", "lower": "lo_q", "upper": "hi_q", "center": "fc_q"},
    {"model": "CRC", "lower": "lo_c", "upper": "hi_c", "center": "fc_c"},
]

p_cmp = plot_forecast_intervals(
    cmp_df,
    interval_specs=interval_specs,
    actual_col="Actual",
    title="Conformal Methods — 90 % Interval Comparison (first 48 steps)",
    y_label="Net Load (kW)",
    fig_size=(820, 380),
)
p_cmp

## Reliability Diagram

Plots empirical coverage against nominal coverage for all three methods.
A well-calibrated method tracks the diagonal; conformal methods are
guaranteed to meet or exceed the target coverage by construction.

In [ ]:
from twiga.core.plot import plot_reliability_diagram

# Compute empirical coverage per nominal level for all three methods
nominal_levels = np.linspace(0.60, 0.99, 20)  # coverage = 1 - alpha

y_true_flat = test_df[data_config.target_feature].values.reshape(-1)

empirical_all = []
for lower_arr, upper_arr in [
    (lower_split.reshape(-1), upper_split.reshape(-1)),
    (lower_cqr.reshape(-1), upper_cqr.reshape(-1)),
    (lower_crc.reshape(-1), upper_crc.reshape(-1)),
]:
    n = min(len(y_true_flat), len(lower_arr))
    y, lo, hi = y_true_flat[:n], lower_arr[:n], upper_arr[:n]
    # For each nominal level, scale the interval by the ratio nominal / 0.9
    # (intervals were computed at alpha=0.1, i.e. 90 % coverage)
    empirical_all.append(
        [
            ((y >= lo - (hi - lo) * (1 - lvl) / 2) & (y <= hi + (hi - lo) * (1 - lvl) / 2)).mean()
            for lvl in nominal_levels
        ]
    )

p_rel = plot_reliability_diagram(
    nominal=nominal_levels,
    empirical=np.array(empirical_all).T,  # shape (n_levels, n_methods)
    groups=["Split Conformal", "CQR", "CRC"],
    title="Reliability Diagram — Conformal Methods",
    fig_size=(480, 440),
)
p_rel

## Wrapping up

**What you did**
- [x] Understood the conformal prediction finite-sample coverage guarantee and the exchangeability assumption
- [x] Calibrated a LightGBM model with Split Conformal using `calibrate()`
- [x] Applied CQR on top of a QR-LightGBM model for locally adaptive intervals
- [x] Applied CRC with an MLPGAMCRCConfig model for heteroscedastic adaptive intervals
- [x] Compared all three methods on coverage (reliability diagram) and sharpness (mean interval width)

**Key takeaways**
1. Conformal prediction produces finite-sample coverage-guaranteed intervals without any distributional assumption — it works with any base model.
2. The calibration set (held-out labeled data) is the core ingredient: nonconformity scores from this set drive the coverage guarantee.
3. Split Conformal is the safest starting point — zero modelling overhead, immediate guarantee.
4. CQR inherits adaptive width from the underlying QR model; CRC trains a secondary residual model for maximum adaptivity.
5. The reliability diagram is the primary diagnostic: all conformal methods should meet or exceed the diagonal; persistent under-coverage signals a violation of the exchangeability assumption.

---

## What's next?

**NB10 — Hyperparameter Optimisation** (`10-hyperparameter-tuning.ipynb`)

Learn how to use Optuna-powered HPO via `forecaster.tune()` to automatically search for optimal model hyperparameters across any model type, with resumable SQLite-backed studies.

In [ ]:
# ruff: noqa: E501, E701, E702
from IPython.display import HTML

_TEAL = "#107591"
_TEAL_MID = "#069fac"
_TEAL_LIGHT = "#e8f5f8"
_TEAL_BEST = "#d0ecf1"
_TEXT_DARK = "#2d3748"
_TEXT_MUTED = "#718096"
_WHITE = "#ffffff"

steps = [
    {
        "num": "08",
        "title": "Quantile Regression",
        "desc": "QR-LightGBM · FPQR — prediction intervals",
        "tags": ["quantile", "pinball loss"],
        "active": False,
    },
    {
        "num": "09",
        "title": "Parametric Distributions",
        "desc": "Normal · Laplace · Gamma heads — NLL training",
        "tags": ["parametric", "NLL"],
        "active": False,
    },
    {
        "num": "10",
        "title": "Conformal Prediction",
        "desc": "CQR · CRC — finite-sample coverage guarantees",
        "tags": ["conformal", "CQR", "CRC", "coverage"],
        "active": True,
    },
    {
        "num": "11",
        "title": "Hyperparameter Tuning",
        "desc": "Optuna-backed HPO · typed search spaces · resumable SQLite",
        "tags": ["optuna", "HPO", "tuning"],
        "active": False,
    },
    {
        "num": "12",
        "title": "Ensemble Strategies",
        "desc": "Mean · median · weighted-mean ensembles",
        "tags": ["ensemble", "weighted", "aggregation"],
        "active": False,
    },
]
track_name = "Probabilistic Track"
footer = 'Next: push coverage further with <span style="color:#107591;font-weight:600;">Hyperparameter Tuning</span> (11) or combine models with <span style="color:#107591;font-weight:600;">Ensemble Strategies</span> (12).'


def _b(t, bg, fg):
    return f'<span style="display:inline-block;background:{bg};color:{fg};font-size:10px;font-weight:600;padding:2px 7px;border-radius:10px;margin:2px 2px 0 0;">{t}</span>'


ch = ""
for i, s in enumerate(steps):
    a = s["active"]
    cb = _TEAL if a else _WHITE
    cbo = _TEAL if a else "#d1ecf1"
    nb = _TEAL_MID if a else _TEAL_LIGHT
    nf = _WHITE if a else _TEAL
    tf = _WHITE if a else _TEXT_DARK
    df = "#cce8ef" if a else _TEXT_MUTED
    bb = "#0d5f75" if a else _TEAL_BEST
    bf = "#b8e4ed" if a else _TEAL
    yh = (
        f'<span style="float:right;background:{_TEAL_MID};color:{_WHITE};font-size:10px;font-weight:700;padding:2px 10px;border-radius:12px;">★ you are here</span>'
        if a
        else ""
    )
    bdg = "".join(_b(t, bb, bf) for t in s["tags"])
    ch += f'<div style="background:{cb};border:2px solid {cbo};border-radius:12px;padding:16px 20px;display:flex;align-items:flex-start;gap:16px;box-shadow:{"0 4px 14px rgba(16,117,145,.25)" if a else "0 1px 4px rgba(0,0,0,.06)"};"><div style="min-width:44px;height:44px;background:{nb};color:{nf};border-radius:50%;display:flex;align-items:center;justify-content:center;font-size:15px;font-weight:800;flex-shrink:0;">{s["num"]}</div><div style="flex:1;"><div style="font-size:15px;font-weight:700;color:{tf};margin-bottom:4px;">{s["title"]}{yh}</div><div style="font-size:12.5px;color:{df};margin-bottom:8px;line-height:1.5;">{s["desc"]}</div><div>{bdg}</div></div></div>'
    if i < len(steps) - 1:
        ch += f'<div style="display:flex;justify-content:center;height:32px;"><svg width="24" height="32" viewBox="0 0 24 32" fill="none"><line x1="12" y1="0" x2="12" y2="24" stroke="{_TEAL_MID}" stroke-width="2" stroke-dasharray="4 3"/><polygon points="6,20 18,20 12,30" fill="{_TEAL_MID}"/></svg></div>'

HTML(
    f'<div style="font-family:Inter,\'Segoe UI\',sans-serif;max-width:640px;margin:8px 0;"><div style="background:linear-gradient(135deg,{_TEAL} 0%,{_TEAL_MID} 100%);border-radius:12px 12px 0 0;padding:14px 20px;display:flex;align-items:center;gap:10px;"><svg width="22" height="22" viewBox="0 0 24 24" fill="none" stroke="{_WHITE}" stroke-width="2"><path d="M12 2L2 7l10 5 10-5-10-5z"/><path d="M2 17l10 5 10-5"/><path d="M2 12l10 5 10-5"/></svg><span style="color:{_WHITE};font-size:14px;font-weight:700;">Twiga Learning Path — {track_name}</span></div><div style="border:2px solid {_TEAL_LIGHT};border-top:none;border-radius:0 0 12px 12px;padding:20px 20px 16px;background:#f9fdfe;display:flex;flex-direction:column;">{ch}<div style="margin-top:16px;font-size:11.5px;color:{_TEXT_MUTED};text-align:center;border-top:1px solid {_TEAL_LIGHT};padding-top:12px;">{footer}</div></div></div>'
)